In [158]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os

In [159]:
path = "./raw_data.csv"

In [160]:
df = pd.read_csv(path)

In [161]:
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
)

if "umber_pets" in df.columns:
    df.rename(columns={"umber_pets": "Number_Pets"}, inplace=True)


In [162]:
df.head()

,CustomerID,Region,Gender,Age,Education_Years,Employment_Years,Job_Category,Retired,Marital_Status,Household_Size,...,Coupon_Redemption,Brand_Tenure_Months,Monthly_Spend_ProductA,Cumulative_Spend_ProductA,Monthly_Spend_ProductB,Cumulative_Spend_ProductB,Monthly_Spend_ProductC,Cumulative_Spend_ProductC,Total_Avg_Monthly_Spend,High_Value_Customer
0,0002-GTOKLU-YVY,5,Male,63,16,3,Sales,No,Married,2,...,0,21,$20.60,$84.92,$147.60,$602.92,$176.50,$790.02,$344.70,1
1,0003-RLTRGE-IW2,1,Male,52,11,14,Professional,No,Unmarried,1,...,0,31,$46.20,$304.28,$0.00,$0.00,$107.25,$746.28,$153.45,0
2,0003-UTGKPR-PRU,2,Male,73,14,11,Professional,No,Unmarried,1,...,0,54,$65.20,$703.88,$0.00,$0.00,$0.00,$0.00,$65.20,0
3,0008-ZIQQOT-SGB,3,Female,40,21,3,Sales,No,Married,3,...,0,1,$8.60,$1.72,$171.20,$34.24,$150.25,$36.06,$330.05,1
4,0012-CIVYLF-839,4,Male,40,12,9,Labor,No,Married,2,...,0,48,$46.60,$495.20,$0.00,$0.00,$0.00,$0.00,$46.60,0


In [163]:

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    if col in ["CustomerID"]:
        #don't modify the value for CustomerID
        continue
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )



In [164]:
df.columns

Index(['CustomerID', 'Region', 'Gender', 'Age', 'Education_Years',
       'Employment_Years', 'Job_Category', 'Retired', 'Marital_Status',
       'Household_Size', 'Household_Income', 'Number_Pets', 'Home_Owner',
       'Car_Ownership', 'Car_Brand', 'Car_Value', 'Commute_Distance',
       'Political_Party', 'Votes', 'Credit_Card', 'Credit_Card_Tenure',
       'Active_Lifestyle', 'TV_Watching_Hours', 'Streaming_Svcs',
       'Wireless_Internet', 'Smart_Phone', 'Twitter_Acct', 'LinkedIn_Acct',
       'Facebook_Acct', 'News_Subscriber', 'Coupon_Redemption',
       'Brand_Tenure_Months', 'Monthly_Spend_ProductA',
       'Cumulative_Spend_ProductA', 'Monthly_Spend_ProductB',
       'Cumulative_Spend_ProductB', 'Monthly_Spend_ProductC',
       'Cumulative_Spend_ProductC', 'Total_Avg_Monthly_Spend',
       'High_Value_Customer'],
      dtype='object')

In [165]:
binary_map = {
    "Yes": 1, "No": 0,
    "Y": 1, "N": 0,
    "True": 1, "False": 0,
    "1": 1, "0": 0
}

binary_cols = [
    "Retired",
    "Home_Owner",
    "Active_Lifestyle",
    "Wireless_Internet",
    "Smart_Phone",
    "Twitter_Acct",
    "LinkedIn_Acct",
    "Facebook_Acct",
    "News_Subscriber",
    "Coupon_Redemption",
    "High_Value_Customer",
    "Streaming_Svcs",
    "Votes",
    "Political_Party"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .map(binary_map)
        )


In [166]:
currency_cols = [
    "Household_Income",
    "Car_Value",
    "Monthly_Spend_ProductA",
    "Cumulative_Spend_ProductA",
    "Monthly_Spend_ProductB",
    "Cumulative_Spend_ProductB",
    "Monthly_Spend_ProductC",
    "Cumulative_Spend_ProductC",
    "Total_Avg_Monthly_Spend"
]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$,]", "", regex=True)
        )

numeric_cols = currency_cols + [
    "Age",
    "Education_Years",
    "Employment_Years",
    "Household_Size",
    "Number_Pets",
    "Commute_Distance"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [167]:

# Start with all rows assumed valid and an empty list of reasons per row
invalid_mask = pd.Series(False, index=df.index)
invalid_reasons = pd.Series([[] for _ in range(len(df))], index=df.index)

# Pattern: for each condition, build a boolean mask `cond`, OR it into `invalid_mask`,
# then append a human-readable reason to `invalid_reasons` for the same rows.

# 1) Nulls in critical fields -> mark invalid
cond = df.isna().any(axis=1)
invalid_mask |= cond
invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Null value'])

# 2) Ensure numeric columns are numeric (coercion created NaN for bad values) -> mark those as invalid
if 'numeric_cols' in locals():
    cond = df[numeric_cols].isna().any(axis=1)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Invalid numeric value'])

# 3) Negative values for Education and Employment years are invalid
if 'Education_Years' in df.columns:
    cond = df['Education_Years'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Education_Years'])
if 'Employment_Years' in df.columns:
    cond = df['Employment_Years'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Employment_Years'])

# 4) Income must be >= 0
if 'Household_Income' in df.columns:
    cond = df['Household_Income'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Household_Income'])

# 5) Household size must be > 0
if 'Household_Size' in df.columns:
    cond = df['Household_Size'] <= 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Non-positive Household_Size'])

# 6) Education years must be less than Age - 5 (assuming people start school around age 5)
if set(['Education_Years','Age']).issubset(df.columns):
    cond = ~(df['Education_Years'] < df['Age'] - 5)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Education_Years >= Age - 5'])

# 6) Employment years cannot exceed plausible range: Employment_Years <= Age - 14
if set(['Employment_Years','Age']).issubset(df.columns):
    cond = ~(df['Employment_Years'] <= (df['Age'] - 14))
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Employment_Years implausible'])

# X) Credit card consistency: Credit_Card == 0 but Credit_Card_Tenure > 0
if set(['Credit_Card','Credit_Card_Tenure']).issubset(df.columns):
    cc = pd.to_numeric(df['Credit_Card'], errors='coerce')
    cc_tenure = pd.to_numeric(df['Credit_Card_Tenure'], errors='coerce')
    cond = (cc == 0) & (cc_tenure > 0)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Credit Card zero with positive tenure'])

# 7) Car value threshold: if provided and below 1000, mark invalid
if 'Car_Value' in df.columns:
    # Coerce to numeric here to ensure values like 'Nocar' become NaN
    car_val = pd.to_numeric(df['Car_Value'], errors='coerce')
    # Treat 0 as 'no car' (do not flag). Only flag positive values below threshold.
    cond = car_val.notna() & (car_val > 0) & (car_val < 1000)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Car_Value below threshold (non-zero)'])

# 9) Cumulative spend checks for Products A, B, C: cumulative >= monthly
for p in ['A','B','C']:
    monthly = f'Monthly_Spend_Product{p}'
    cumulative = f'Cumulative_Spend_Product{p}'
    if (monthly in df.columns) and (cumulative in df.columns):
        cond = df[cumulative].isna() | df[monthly].isna() | (df[cumulative] < df[monthly])
        invalid_mask |= cond
        invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + [f'Cumulative < Monthly for Product{p}'])

# 10) Remove duplicate customer records: keep first, mark subsequent duplicates as invalid
if 'CustomerID' in df.columns:
    dup_mask = df.duplicated(subset='CustomerID', keep='first')
    invalid_mask |= dup_mask
    invalid_reasons.loc[dup_mask] = invalid_reasons.loc[dup_mask].apply(lambda lst: lst + ['Duplicate CustomerID'])

# Build invalid and cleaned DataFrames
invalid_df = df[invalid_mask].copy()
# Attach a single string column summarizing reasons; empty string if none
invalid_df['Invalid_Reasons'] = invalid_reasons[invalid_mask].apply(lambda lst: '; '.join(lst) if lst else '')
cleaned_df = df[~invalid_mask].copy()

# Ensure cleaned_df has no duplicates (keep first) and drop them if present
if 'CustomerID' in cleaned_df.columns:
    cleaned_df = cleaned_df.drop_duplicates(subset='CustomerID', keep='first')


In [168]:
cleaned_df.head()

,CustomerID,Region,Gender,Age,Education_Years,Employment_Years,Job_Category,Retired,Marital_Status,Household_Size,...,Coupon_Redemption,Brand_Tenure_Months,Monthly_Spend_ProductA,Cumulative_Spend_ProductA,Monthly_Spend_ProductB,Cumulative_Spend_ProductB,Monthly_Spend_ProductC,Cumulative_Spend_ProductC,Total_Avg_Monthly_Spend,High_Value_Customer
0,0002-GTOKLU-YVY,5,Male,63,16,3,Sales,0,Married,2,...,0,21,20.6,84.92,147.6,602.92,176.50,790.02,344.70,1
1,0003-RLTRGE-IW2,1,Male,52,11,14,Professional,0,Unmarried,1,...,0,31,46.2,304.28,0.0,0.00,107.25,746.28,153.45,0
2,0003-UTGKPR-PRU,2,Male,73,14,11,Professional,0,Unmarried,1,...,0,54,65.2,703.88,0.0,0.00,0.00,0.00,65.20,0
4,0012-CIVYLF-839,4,Male,40,12,9,Labor,0,Married,2,...,0,48,46.6,495.20,0.0,0.00,0.00,0.00,46.60,0
5,0014-DOIOFX-LXB,2,Female,37,12,10,Crafts,0,Married,4,...,0,59,93.6,1175.76,0.0,0.00,0.00,0.00,93.60,0


In [169]:
invalid_df.head()

,CustomerID,Region,Gender,Age,Education_Years,Employment_Years,Job_Category,Retired,Marital_Status,Household_Size,...,Brand_Tenure_Months,Monthly_Spend_ProductA,Cumulative_Spend_ProductA,Monthly_Spend_ProductB,Cumulative_Spend_ProductB,Monthly_Spend_ProductC,Cumulative_Spend_ProductC,Total_Avg_Monthly_Spend,High_Value_Customer,Invalid_Reasons
3,0008-ZIQQOT-SGB,3,Female,40,21,3,Sales,0,Married,3,...,1,8.6,1.72,171.2,34.24,150.25,36.06,330.05,1,Cumulative < Monthly for ProductA; Cumulative ...
8,0016-XBFXXW-2G0,4,Female,19,14,0,Professional,0,Married,3,...,22,22.6,101.48,107.8,435.16,0.00,0.00,130.40,0,Education_Years >= Age - 5
11,0022-QFOJNN-XSH,3,Male,39,13,4,Sales,0,Married,3,...,3,16.8,11.24,128.4,50.84,110.00,38.28,255.20,0,Cumulative < Monthly for ProductA; Cumulative ...
14,0023-WKLZQN-M6X,5,Female,35,17,2,Sales,0,Married,2,...,2,10.8,4.56,0.0,0.00,130.75,42.30,141.55,0,Cumulative < Monthly for ProductA; Cumulative ...
15,0024-UQUVXG-79W,1,Male,26,21,0,Sales,0,Married,3,...,9,20.6,47.04,252.2,379.80,399.75,790.08,672.55,1,Education_Years >= Age - 5


In [170]:
# Create timestamped output directory: outputs/<yy-mm-dd-hh>/
ts = datetime.now().strftime('%y-%m-%d-%H')
outdir = Path('outputs') / ts
outdir.mkdir(parents=True, exist_ok=True)

In [171]:
# Save outputs as Excel workbooks
cleaned_df.to_excel(outdir / 'customer_data_cleaned.xlsx', index=False)
invalid_df.to_excel(outdir / 'invalid.xlsx', index=False)

print('Saved', cleaned_df.shape[0], 'clean rows and', invalid_df.shape[0], 'invalid rows to', outdir)

Saved 4363 clean rows and 637 invalid rows to outputs\26-04-14-16
